# 📘 Customer Churn Analysis – Telecom Industry
**Objective:** Predict churn and identify retention strategies in a telecom setting.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import eli5 # type: ignore
from eli5.sklearn import PermutationImportance # type: ignore
import seaborn as sns
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'eli5'

In [ ]:
conn = sqlite3.connect('telecom_churn.db')

query = '''
SELECT c.customer_id,
       c.plan_type,
       COALESCE(cd.total_call_minutes, 0) AS total_call_minutes,
       COALESCE(cm.total_complaints, 0) AS total_complaints,
       COALESCE(r.recharge_count, 0) AS recharge_count,
       r.last_recharge_date,
       l.churned
FROM customers c
LEFT JOIN (
    SELECT customer_id, SUM(duration_minutes) AS total_call_minutes
    FROM calls GROUP BY customer_id
) cd ON c.customer_id = cd.customer_id
LEFT JOIN (
    SELECT customer_id, COUNT(*) AS total_complaints
    FROM complaints GROUP BY customer_id
) cm ON c.customer_id = cm.customer_id
LEFT JOIN (
    SELECT customer_id, COUNT(*) AS recharge_count, MAX(recharge_date) AS last_recharge_date
    FROM recharges GROUP BY customer_id
) r ON c.customer_id = r.customer_id
LEFT JOIN churn_labels l ON c.customer_id = l.customer_id;
'''

df = pd.read_sql_query(query, conn)
df['last_recharge_date'] = pd.to_datetime(df['last_recharge_date'])
df['days_since_last_recharge'] = (datetime(2025, 4, 25) - df['last_recharge_date']).dt.days
df.head()


In [ ]:
df = df.drop(columns=['customer_id', 'last_recharge_date'])
df['plan_type'] = df['plan_type'].astype('category').cat.codes
df.fillna(0, inplace=True)

X = df.drop('churned', axis=1)
y = df['churned']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
perm = PermutationImportance(model, random_state=1).fit(X_test, y_test)
eli5.show_weights(perm, feature_names=X.columns.tolist())


In [ ]:
df['churn_prob'] = model.predict_proba(X_scaled)[:, 1]
df['segment'] = pd.cut(df['churn_prob'], bins=[0, 0.3, 0.7, 1], labels=['Loyal', 'Dormant', 'At Risk'])
df['segment'].value_counts()


In [ ]:
sns.countplot(data=df, x='segment', palette='coolwarm')
plt.title("Customer Segments Based on Churn Probability")
plt.show()


In [ ]:
df.to_csv('churn_results_with_segments.csv', index=False)
